In [10]:
from __future__ import annotations

import json
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score, 
    recall_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [11]:
DATA_PATH = './data/datatran2025.csv'

In [12]:
# Funções auxiliares para manipular o dataset

def load_dataset(path: Path, sep: str = None, encoding: str = "latin1") -> pd.DataFrame:
    df = pd.read_csv(path, sep=sep, engine="python", encoding=encoding)
    return df


def preprocess_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for column in [
        "mortos",
        "feridos_leves",
        "feridos_graves",
        "ilesos",
        "feridos",
        "ignorados",
    ]:
        df[f"teve_{column}"] = df[column] > 0

    return df



In [13]:
df = preprocess_dataset(load_dataset(DATA_PATH))

df

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,longitude,regional,delegacia,uop,teve_mortos,teve_feridos_leves,teve_feridos_graves,teve_ilesos,teve_feridos,teve_ignorados
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,...,"-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,False,True,False,False,True,True
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,...,"-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,True,True,False,True,True,True
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,...,"-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,False,True,False,True,True,False
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,...,"-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,False,True,False,True,True,False
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,...,"-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,False,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72524,757462,2025-12-20,sábado,13:10:00,MG,381,"279,5",JAGUARACU,Acessar a via sem observar a presença dos outr...,Colisão transversal,...,"-42,7611",SPRF-MG,DEL03-MG,UOP02-DEL03-MG,False,True,False,True,True,False
72525,757492,2025-09-27,sábado,13:50:00,SC,282,381,HERVAL DOESTE,Conversão proibida,Colisão transversal,...,"-51,50077",SPRF-SC,DEL07-SC,UOP05-DEL07-SC,False,True,True,False,True,True
72526,757593,2025-12-14,domingo,13:50:00,PE,104,62,CARUARU,Ausência de reação do condutor,Colisão transversal,...,"-35,97546",SPRF-PE,DEL02-PE,UOP01-DEL02-PE,False,True,False,True,True,False
72527,758175,2025-12-15,segunda-feira,15:50:00,SC,101,130,CAMBORIU,Condutor deixou de manter distância do veículo...,Colisão traseira,...,"-48,6609548",SPRF-SC,DEL04-SC,UOP03-DEL04-SC,False,False,False,True,False,False


In [14]:
boolean_cols = df.select_dtypes(include="bool").columns.tolist()

bool_counts = pd.DataFrame(
    {
        "True": df[boolean_cols].sum(),
        "False": (~df[boolean_cols]).sum(),
    }
).astype(int)

display(bool_counts)

categorical_columns = [
    "tipo_acidente",
    "classificacao_acidente",
    "causa_acidente",
    "condicao_metereologica",
    "tipo_pista",
    "tracado_via",
]

unique_values = {
    column: sorted(df[column].dropna().unique().tolist())
    for column in categorical_columns
    if column in df.columns
}

unique_values_df = pd.DataFrame(
    {
        "coluna": list(unique_values.keys()),
        "valores_unicos": list(unique_values.values()),
    }
 )

display(unique_values_df)

,True,False
teve_mortos,5210,67319
teve_feridos_leves,45966,26563
teve_feridos_graves,16354,56175
teve_ilesos,45198,27331
teve_feridos,58042,14487
teve_ignorados,19857,52672


,coluna,valores_unicos
0,tipo_acidente,"[Atropelamento de Animal, Atropelamento de Ped..."
1,classificacao_acidente,"[Com Vítimas Fatais, Com Vítimas Feridas, Sem ..."
2,causa_acidente,[Acessar a via sem observar a presença dos out...
3,condicao_metereologica,"[Chuva, Céu Claro, Garoa/Chuvisco, Ignorado, N..."
4,tipo_pista,"[Dupla, Múltipla, Simples]"
5,tracado_via,"[Aclive, Aclive;Curva, Aclive;Curva;Em Obras, ..."
